# Week 4 — Chunking 크기 격자 실험 v2 (Chunking Experiments)

## 목표
데이터 진단(week4_data_analysis) 결과를 근거로, **언어별 chunk 크기 조합을 격자(grid)로 비교**해 한·영 혼합 유방암 가이드라인에 가장 잘 맞는 크기를 RAGAS로 수치화한다.

v1(A/B/C 임의 비교)에서 "aggregate 점수가 특정 문항에 좌우되고 RAGAS judge 변동성이 크다"는 한계를 확인하여, v2에서는 (1) 평가셋 20문항으로 확대, (2) judge를 Sonnet 5로 상향, (3) 크기 변수를 언어별 격자로 격리해 재설계했다.

선행 노트북: `week4_data_analysis.ipynb` (데이터 진단)

## 크기 산정 근거 (데이터 진단)
- 한국어 페이지 평균 약 1843자, 영어 페이지 평균 약 1584자(추정: 396토큰 × 4자/토큰)
- 목표 조각 수별 크기: `step = 페이지글자수 / 조각수`, `chunk_size = step / 0.85`, `overlap = 15%`
- 2조각(≈1000자 이상)은 v1에서 정밀도 저하가 확인되어 제외 → **4조각·3조각만** 비교

| 언어 | 4조각 | 3조각 |
|---|---|---|
| 한국어 | 540 / 80 | 720 / 110 |
| 영어 | 470 / 70 | 620 / 90 |

## 비교 전략 — 2×2 격자 (클린 off, 크기만 격리)
| 전략 | 한국어 | 영어 | 의도 |
|---|---|---|---|
| **G1_ko4_en4** | 540/80 | 470/70 | 둘 다 잘게(4조각) |
| **G2_ko4_en3** | 540/80 | 620/90 | 한국어 4 · 영어 3 |
| **G3_ko3_en4** | 720/110 | 470/70 | 한국어 3 · 영어 4 |
| **G4_ko3_en3** | 720/110 | 620/90 | 둘 다 크게(3조각) |

노이즈 제거(클린)는 크기 변수와 섞이지 않도록 이 격자에서 제외한다. G1~G4 중 최적을 확인한 뒤, 그 크기에 클린만 얹어 별도 config로 효과를 측정한다(방법1, 2단계).

## 공통 조건 (크기 외 변수 통제)
- 평가셋: `golden_set_v1.csv` (20문항)
- 로더: PyMuPDFLoader (baseline과 동일) / 임베딩: multilingual-e5-base
- 생성: gpt-4o-mini (1회) / 채점: **Claude Sonnet 5** (1회, median 없음)
- retriever TOP_K, 프롬프트 등은 baseline과 동일

---
## 1. 경로 / 설정 (CONFIG)

생성 gpt-4o-mini / 채점 Sonnet 5 고정. 크기 격자 4개(G1~G4)를 STRATEGIES에 정의한다.

In [1]:
from pathlib import Path
import os, json
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_ROOT = PROJECT_ROOT / "data" / "vector_store"
for p in [DATA_PROCESSED, DATA_EVAL, VECTOR_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env에 없거나 로드 안 됨"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 .env에 없거나 로드 안 됨"

# ---- 고정값: 생성 gpt-4o-mini / 채점 Sonnet 5 (변동성 완화 위해 Haiku에서 상향) ----
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
GEN_MODEL = "gpt-4o-mini"       # 답변 생성: OpenAI (1회)
JUDGE_MODEL = "claude-sonnet-5" # RAGAS 채점: Anthropic Sonnet 5 (1회, median 없음)
EMBED_DEVICE = "cpu"
TOP_K = 5

# ---- 크기 격자 실험 (언어별 chunk_size, 클린 없음) ----
# 데이터 진단 근거: 한국어 페이지 ~1843자, 영어 ~1584자(추정)
#   4조각 → step≈페이지/4,  chunk_size=step/0.85, overlap=15%
#   한국어: 4조각 540/80, 3조각 720/110  |  영어: 4조각 470/70, 3조각 620/90
# 2조각(≈1000+)은 정밀도 저하가 이미 확인되어 제외
STRATEGIES = {
    "G1_ko4_en4": {"kind": "lang_size",
                   "size_by_lang": {"ko": 540, "en": 470, "unknown": 500},
                   "overlap_by_lang": {"ko": 80, "en": 70, "unknown": 75}},
    "G2_ko4_en3": {"kind": "lang_size",
                   "size_by_lang": {"ko": 540, "en": 620, "unknown": 580},
                   "overlap_by_lang": {"ko": 80, "en": 90, "unknown": 85}},
    "G3_ko3_en4": {"kind": "lang_size",
                   "size_by_lang": {"ko": 720, "en": 470, "unknown": 600},
                   "overlap_by_lang": {"ko": 110, "en": 70, "unknown": 90}},
    "G4_ko3_en3": {"kind": "lang_size",
                   "size_by_lang": {"ko": 720, "en": 620, "unknown": 670},
                   "overlap_by_lang": {"ko": 110, "en": 90, "unknown": 100}},
}
# 클린(노이즈 제거)은 1~4 최적 확인 후 별도 config로 추가 (방법1, 2단계)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GEN/JUDGE:", GEN_MODEL, "/", JUDGE_MODEL)
print("strategies:", list(STRATEGIES.keys()))

PROJECT_ROOT: /Users/jian/Documents/rag-agent-portfolio
GEN/JUDGE: gpt-4o-mini / claude-sonnet-5
strategies: ['G1_ko4_en4', 'G2_ko4_en3', 'G3_ko3_en4', 'G4_ko3_en3']


---
## 2. 공통 — PDF 로드 (PyMuPDFLoader)

모든 전략이 동일한 원본 Document(PyMuPDFLoader)에서 출발해야 크기 비교가 공정하다. 로더 비교는 별도 실험(v3)에서 다룬다.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
import re

manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
else:
    meta_lookup = {}


def load_all_pdfs(pdf_root: Path) -> list:
    all_docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        loader = PyMuPDFLoader(str(pdf_path))
        docs = loader.load()
        extra = meta_lookup.get(pdf_path.name, {})
        for d in docs:
            d.metadata.update(
                {
                    "filename": pdf_path.name,
                    "source_folder": pdf_path.parent.name,
                    "org": extra.get("org", pdf_path.parent.name),
                    "title": extra.get("title", pdf_path.stem),
                    "language": extra.get("language", "unknown"),
                    "doc_type": extra.get("doc_type", "unknown"),
                    "priority": extra.get("priority", "unknown"),
                }
            )
        all_docs.extend(docs)
    return all_docs


documents = load_all_pdfs(DATA_RAW / "pdf")
print(f"로드된 페이지 Document 수: {len(documents)}")


# language 메타가 unknown인 경우 한글 비율로 fallback
def guess_lang(text: str) -> str:
    kr = len(re.findall(r"[\uac00-\ud7a3]", text))
    return "ko" if kr > 20 else "en"


for d in documents:
    if d.metadata.get("language") in (None, "unknown", "?"):
        d.metadata["language"] = guess_lang(d.page_content)

로드된 페이지 Document 수: 796


---
## 3. 노이즈 클린 유틸 (참고용 — 이번 격자에서는 미사용)

이번 v2 크기 격자(G1~G4)는 클린을 적용하지 않는다(크기 변수 격리). 아래 함수는 최적 크기 확정 후 클린 config를 추가할 때 사용한다.

- **페이지번호**: 숫자 줄 전부가 아니라 **페이지 맨위/맨아래 위치**의 숫자 줄만 제거(표 안 숫자 보존)
- **머리말/꼬리말**: 전체 페이지의 **30% 이상 반복**되는 줄만 제거(본문 오제거 방지)

In [3]:
from collections import Counter

PAGE_NUM_PATTERNS = [
    re.compile(r"^\d{1,4}$"),
    re.compile(r"^-\s?\d{1,4}\s?-$"),
    re.compile(r"^Page\s+\d+", re.IGNORECASE),
    re.compile(r"^\d+\s*/\s*\d+$"),
]


def detect_running_headers(
    doc_pages: list, min_ratio: float = 0.30, max_len: int = 50
) -> set:
    """한 문서 내에서 전체 페이지의 min_ratio 이상에 반복되는 짧은 줄 = 머리말/꼬리말 후보."""
    n_pages = len(doc_pages)
    if n_pages < 4:
        return set()
    counter = Counter()
    for text in doc_pages:
        lines = {l.strip() for l in text.split("\n") if 1 <= len(l.strip()) <= max_len}
        for l in lines:
            counter[l] += 1
    threshold = max(3, int(n_pages * min_ratio))
    return {l for l, c in counter.items() if c >= threshold}


def strip_page_number_lines(text: str) -> str:
    """페이지 맨위/맨아래 위치의 숫자 전용 줄만 제거 (표 안 숫자 보존)."""
    lines = text.split("\n")
    nonempty_idx = [i for i, l in enumerate(lines) if l.strip()]
    if not nonempty_idx:
        return text
    edge = set(nonempty_idx[:2] + nonempty_idx[-2:])  # 앞 2줄, 끝 2줄
    out = []
    for i, l in enumerate(lines):
        s = l.strip()
        if i in edge and any(p.match(s) for p in PAGE_NUM_PATTERNS):
            continue
        out.append(l)
    return "\n".join(out)


def clean_documents(docs: list) -> list:
    """filename 단위로 머리말 감지 -> 머리말·페이지번호 제거."""
    by_file = {}
    for d in docs:
        by_file.setdefault(d.metadata.get("filename", "?"), []).append(d)

    cleaned = []
    total_removed = 0
    for fname, pages in by_file.items():
        headers = detect_running_headers([p.page_content for p in pages])
        for d in pages:
            text = strip_page_number_lines(d.page_content)
            kept = []
            for l in text.split("\n"):
                if l.strip() in headers:
                    total_removed += 1
                    continue
                kept.append(l)
            cleaned.append(
                Document(page_content="\n".join(kept), metadata=dict(d.metadata))
            )
    print(f"제거된 머리말/꼬리말 줄 수(누적): {total_removed}")
    return cleaned


# 감지된 머리말 샘플 확인 (ESMO)
sample_pages = [
    d.page_content
    for d in documents
    if d.metadata.get("filename", "").startswith("esmo")
]
print("ESMO 머리말/꼬리말 후보:", detect_running_headers(sample_pages))

ESMO 머리말/꼬리말 후보: {'유방암', '환자를 위한 ESMO 안내서'}


---
## 4. 청킹 전략 정의 (언어별 크기 격자)

`lang_size` 방식: 문서의 language 메타데이터에 따라 한국어/영어에 서로 다른 chunk_size를 적용한다. 클린은 하지 않는다.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

SEPARATORS = ["\n\n", "\n", ". ", " ", ""]


def make_splitter(size: int, overlap: int) -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap,
        separators=SEPARATORS,
        length_function=len,
    )


def _split_by_lang(docs, size_by_lang, overlap_by_lang):
    """언어별로 다른 chunk_size 적용해 분할."""
    out = []
    for lang in set(d.metadata.get("language", "unknown") for d in docs):
        size = size_by_lang.get(lang, size_by_lang["unknown"])
        overlap = overlap_by_lang.get(lang, overlap_by_lang["unknown"])
        sub = [d for d in docs if d.metadata.get("language", "unknown") == lang]
        out.extend(make_splitter(size, overlap).split_documents(sub))
    return out


def build_chunks(strategy_name: str) -> list:
    cfg = STRATEGIES[strategy_name]
    # 크기 격자 실험: 언어별 크기 적용, 노이즈 제거는 하지 않음 (크기 변수만 격리)
    if cfg["kind"] == "lang_size":
        return _split_by_lang(documents, cfg["size_by_lang"], cfg["overlap_by_lang"])
    # (참고용) 통일 크기
    if cfg["kind"] == "recursive":
        return make_splitter(cfg["chunk_size"], cfg["chunk_overlap"]).split_documents(documents)
    # (참고용) 언어별 크기 + 노이즈 제거 — 5번째 클린 config에서 사용
    if cfg["kind"] == "cleaned":
        cleaned = clean_documents(documents)
        return _split_by_lang(cleaned, cfg["size_by_lang"], cfg["overlap_by_lang"])
    raise ValueError(strategy_name)


chunk_store = {}
for name in STRATEGIES:
    ck = build_chunks(name)
    chunk_store[name] = ck
    avg = sum(len(c.page_content) for c in ck) / max(len(ck), 1)
    print(f"{name:12s} chunk수={len(ck):5d}  평균길이(문자)={avg:.0f}")

G1_ko4_en4   chunk수= 3091  평균길이(문자)=427
G2_ko4_en3   chunk수= 2773  평균길이(문자)=477
G3_ko3_en4   chunk수= 2753  평균길이(문자)=487
G4_ko3_en3   chunk수= 2435  평균길이(문자)=551


---
## 5. 인덱싱 / retriever 빌더 (전략별 별도 컬렉션)

컬렉션명은 `week4v2_*`로, v1 인덱스와 충돌하지 않게 분리한다.

In [5]:
import os
from huggingface_hub import snapshot_download

# 이전 실행에서 켜진 오프라인 모드 해제 + 네트워크 확인 타임아웃 (임베딩 hang 방지)
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# 로컬 캐시에 완전한 snapshot이 있으면 그 경로를 직접 사용 (repo_id 조회로 인한 hang 회피)
try:
    model_dir = snapshot_download(repo_id=EMBEDDING_MODEL, local_files_only=True)
    print("로컬 캐시에서 모델 snapshot 확인:", model_dir)
except Exception as e:
    print("로컬 캐시 불완전 -> 다운로드 진행:", repr(e))
    model_dir = snapshot_download(
        repo_id=EMBEDDING_MODEL, local_files_only=False, max_workers=1
    )
    print("모델 다운로드 완료:", model_dir)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import chromadb

embeddings = HuggingFaceEmbeddings(
    model_name=model_dir,  # repo_id가 아니라 실제 로컬 snapshot 경로 사용
    model_kwargs={"device": EMBED_DEVICE},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16},
)
print("embedding 로드 완료")


def build_retriever(strategy_name: str):
    """전략별 Chroma 컬렉션 생성(없으면) 또는 재사용."""
    vdir = VECTOR_ROOT / f"week4v2_{strategy_name}"
    vdir.mkdir(parents=True, exist_ok=True)
    coll = f"breast_rag_week4v2_{strategy_name}"
    client = chromadb.PersistentClient(path=str(vdir))
    if coll in [c.name for c in client.list_collections()]:
        vs = Chroma(
            collection_name=coll,
            embedding_function=embeddings,
            persist_directory=str(vdir),
        )
        print(f"  {strategy_name}: 기존 컬렉션 재사용 ({vs._collection.count()}개)")
    else:
        ck = chunk_store[strategy_name]
        print(f"  {strategy_name}: 신규 인덱싱 {len(ck)}개 ... (CPU면 시간 소요)")
        vs = Chroma.from_documents(
            documents=ck,
            embedding=embeddings,
            collection_name=coll,
            persist_directory=str(vdir),
        )
    return vs.as_retriever(search_kwargs={"k": TOP_K})

로컬 캐시에서 모델 snapshot 확인: /Users/jian/.cache/huggingface/hub/models--intfloat--multilingual-e5-base/snapshots/d128750597153bb5987e10b1c3493a34e5a4502a
embedding 로드 완료


---
## 6. RAG 체인 (baseline과 동일 프롬프트)

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=GEN_MODEL, temperature=0)

RAG_PROMPT = ChatPromptTemplate.from_template(
    """당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]"""
)


def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        m = d.metadata
        parts.append(
            f"[{i}] 출처: {m.get('org','?')} / {m.get('title','?')} / p.{m.get('page','?')}\n{d.page_content}"
        )
    return "\n\n---\n\n".join(parts)


def make_ask(retriever):
    def ask(question: str):
        docs = retriever.invoke(question)
        prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
        answer = (llm | StrOutputParser()).invoke(prompt)
        return {
            "question": question,
            "answer": answer,
            "contexts": [d.page_content for d in docs],
        }

    return ask

---
## 7. 평가 질문 세트 (golden_set_v1, 30문항)

v1의 10문항에서 30문항으로 확대한 세트. 한국어 질문 20문항, 영어 질문 10문항으로 구성되어 있다. RAGAS judge 변동성이 소수 문항에 좌우되는 것을 완화하기 위함과 좀 더 정확한 청킹 실험을 위해 영어 버전도 추가했다

In [7]:
import pandas as pd

golden_path = DATA_EVAL / "golden_set_v1.csv"
if golden_path.exists():
    df_golden = pd.read_csv(golden_path)
    print(f"golden_set_v1 로드: {len(df_golden)}문항")
else:
    raise FileNotFoundError("golden_set_v1.csv 없음 — data/eval/ 에 golden_set_v1.csv를 넣으세요.")
golden = df_golden.to_dict("records")
df_golden[["question"]]

golden_set_v1 로드: 30문항


,question
0,유방암 검진은 몇 살부터 받는 것이 권장되나요?
1,유방암의 주요 위험 요인은 무엇인가요?
2,HER2 양성 유방암이란 무엇인가요?
3,유방암 1기와 2기의 차이는 무엇인가요?
4,DCIS(유관상피내암)는 침윤성 유방암과 어떻게 다른가요?
5,항호르몬제 치료는 어떤 환자에게 사용되나요?
6,유방절제술 후 재건수술에는 어떤 방법이 있나요?
7,BRCA 유전자 검사는 누구에게 권장되나요?
8,전이성 유방암인 4기의 일반적인 치료 목표는 무엇인가요?
9,유방암 환자가 식이요법에서 주의할 점은 무엇인가요?


---
## 8. 전략별 실행 + RAGAS 평가

크기 격자 4개(G1~G4)를 각각 인덱싱 → 생성(gpt-4o-mini) → 채점(Sonnet 5, 1회).
- 20문항 × 4전략이라 CPU 인덱싱 + 채점에 시간이 걸린다(전략당 수 분~십수 분).
- 채점은 1회만(median 없음). 순위가 애매하면 이후 반복 측정을 추가한다.
- GPU가 있으면 CONFIG의 `EMBED_DEVICE="cuda"`로 인덱싱을 단축할 수 있다.

In [8]:
import nest_asyncio

nest_asyncio.apply()  # Jupyter async 충돌로 인한 RAGAS NaN 방지

from ragas import evaluate, EvaluationDataset
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from tqdm import tqdm
from langchain_anthropic import ChatAnthropic

class ChatAnthropicNoTemp(ChatAnthropic):
    def __setattr__(self, name, value):
        if name == "temperature":
            value = None
        super().__setattr__(name, value)

ragas_llm = LangchainLLMWrapper(ChatAnthropicNoTemp(model=JUDGE_MODEL, max_tokens=4096))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)
METRICS = [faithfulness, answer_relevancy, context_precision]
METRIC_COLS = ["faithfulness", "answer_relevancy", "context_precision"]


def run_strategy(name: str) -> pd.DataFrame:
    retriever = build_retriever(name)
    ask = make_ask(retriever)
    rows = []
    for g in tqdm(golden, desc=f"RAG[{name}]"):
        r = ask(g["question"])
        rows.append(
            {
                "question": r["question"],
                "answer": r["answer"],
                "contexts": r["contexts"],
                "ground_truth": g.get("ground_truth", ""),
            }
        )
    # ragas 0.2 스키마: user_input / response / retrieved_contexts / reference
    # (옛 question/answer/contexts/ground_truth로 넣으면 reference 미전달 -> context_precision=0)
    samples = [
        {
            "user_input": r["question"],
            "response": r["answer"],
            "retrieved_contexts": r["contexts"],
            "reference": r["ground_truth"],
        }
        for r in rows
    ]
    ds = EvaluationDataset.from_list(samples)
    scores = evaluate(dataset=ds, metrics=METRICS, llm=ragas_llm, embeddings=ragas_emb)
    df = scores.to_pandas()
    df.to_csv(
        DATA_PROCESSED / f"week4_ragas_{name}_v2.csv", index=False, encoding="utf-8-sig"
    )
    return df


score_tables = {}
for name in STRATEGIES:
    score_tables[name] = run_strategy(name)
    print(f"{name}: 완료")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  G1_ko4_en4: 기존 컬렉션 재사용 (3091개)


RAG[G1_ko4_en4]: 100%|██████████| 30/30 [01:20<00:00,  2.69s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Exception raised in Job[21]: LLMDidNotFinishException(The LLM generation was not completed. Please increase the max_tokens and try again.)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


...

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


G3_ko3_en4: 완료
  G4_ko3_en3: 기존 컬렉션 재사용 (2435개)


RAG[G4_ko3_en3]: 100%|██████████| 30/30 [01:21<00:00,  2.73s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

G4_ko3_en3: 완료


---
## 9. 비교 결과표

델타는 첫 전략(G1)을 기준으로 계산한다. Context Precision을 중심으로 보되, 절대값보다 전략 간 상대 순위를 신뢰한다(judge 변동성 고려).

In [9]:
rows = []
for name in STRATEGIES:
    df = score_tables[name]
    ck = chunk_store[name]
    row = {
        "전략": name,
        "chunk수": len(ck),
        "평균길이": round(sum(len(c.page_content) for c in ck) / max(len(ck), 1)),
    }
    for col in METRIC_COLS:
        row[col] = round(df[col].mean(), 4) if col in df.columns else None
    rows.append(row)

df_compare = pd.DataFrame(rows)

# baseline 대비 델타 추가
base_name = list(STRATEGIES)[0]  # 첫 전략을 델타 기준으로
base = df_compare[df_compare["전략"] == base_name].iloc[0]
for col in METRIC_COLS:
    df_compare[col + "_delta"] = (df_compare[col] - base[col]).round(4)

df_compare.to_csv(
    DATA_PROCESSED / "week4_chunking_comparison_v2.csv", index=False, encoding="utf-8-sig"
)
print("저장: week4_chunking_comparison_v2.csv")
df_compare

저장: week4_chunking_comparison_v2.csv


,전략,chunk수,평균길이,faithfulness,answer_relevancy,context_precision,faithfulness_delta,answer_relevancy_delta,context_precision_delta
0,G1_ko4_en4,3091,427,0.6242,0.9022,0.6449,0.0000,0.0000,0.0000
1,G2_ko4_en3,2773,477,0.6561,0.9039,0.6773,0.0319,0.0017,0.0324
2,G3_ko3_en4,2753,487,0.6202,0.9027,0.6503,-0.0040,0.0005,0.0054
3,G4_ko3_en3,2435,551,0.6122,0.9050,0.6344,-0.0120,0.0028,-0.0105


---
## 10. 에러 케이스 분석 (전략별 최저 context_precision 문항)

언어별로 점수가 낮은 문항은 실제 검색된 chunk를 열어 "진짜 검색 실패인지, judge 변동/맥락 손실인지" 확인한다.

In [10]:
import re

def _lang(q):
    return "KO" if re.search("[가-힣]", str(q)) else "EN"

for name in STRATEGIES:
    df = score_tables[name]
    if "context_precision" not in df.columns:
        continue
    # 최신 RAGAS 컬럼명 대응
    COL_Q = "user_input" if "user_input" in df.columns else "question"
    COL_C = "retrieved_contexts" if "retrieved_contexts" in df.columns else "contexts"

    # 언어 라벨 부여
    df = df.copy()
    df["_lang"] = df[COL_Q].apply(_lang)

    print("=" * 64)
    print(f"[{name}] 언어별 context_precision 최저 문항")

    for lang in ["KO", "EN"]:
        sub = df[df["_lang"] == lang]
        if sub.empty:
            continue
        # 언어별 평균도 같이 출력 (언어별 청킹 해석용)
        print("-" * 64)
        print(f"  [{lang}] 문항 {len(sub)}개 | 평균 context_precision={sub['context_precision'].mean():.3f}")
        worst = sub.nsmallest(2, "context_precision")
        for _, r in worst.iterrows():
            fp = r.get("faithfulness", float("nan"))
            print(f"    질문: {r[COL_Q]}")
            print(f"    context_precision={r['context_precision']:.3f}  faithfulness={fp:.3f}")
            ctxs = r[COL_C] if isinstance(r[COL_C], list) else []
            for i, c in enumerate(ctxs[:2], 1):
                print(f"      [{i}] {str(c)[:150].replace(chr(10), ' ')} ...")
            print()

[G1_ko4_en4] 언어별 context_precision 최저 문항
----------------------------------------------------------------
  [KO] 문항 20개 | 평균 context_precision=0.624
    질문: 유방암 1기와 2기의 차이는 무엇인가요?
    context_precision=0.000  faithfulness=0.000
      [1] 2023 The 10 th Korean Clinical Practice Guideline for Breast Cancer | 41 제2장 조기 유방암 제3장 재발 및 전이성 유방암 제4장 유전성 유방암  제1장 비침습 유방암 제1장 비침습 유방암: 관상피내암과 소엽상피 ...
      [2] 매우 드물지만 남성에서도 발생하며 이는 전체 유방암 사례의 약 1%를 차지합니다. 유방암 진단 •	 유방암의 가장 흔한 증상은 응어리의 존재, 유두의 변화, 유두의 분비물 또는 유방 피부의 변화와  같은 유방의 변화입니다. •	 유방암에 대한 초기 검사는 신체 검사,  ...

    질문: 유방암 환자가 식이요법에서 주의할 점은 무엇인가요?
    context_precision=0.000  faithfulness=0.714
      [1] 국민 암예방 수칙 실천지침 _ 유방암 8 9  유방암을 예방할 수 있는 방법을  구체적으로 알아보도록 하겠습니다. ▶ 하나!  건강 체중 유지하기 두번째 체중 감량과 건강 체중 유지를 위한 좋은 식사 습관 유지 자신이 무의식적으로 어떤 것을 어떻게 먹고 있는지 알게 되 ...
      [2] 58 유방암 건강 관리 유방암 치료를 받은 후에는 매우 피곤하고 감정적이 될 수 있습니다. 신체가 회복할 시간을 가지면서  충분한 휴식을 취하십시오. 하지만 기분이 좋다면 활동을 제한할 이유가 없습니다. 자신을 잘  돌보면서 가족 활동, 일 또는 직업적 역할을 포함한  ...

-------------------------

## 11. 클렌징(노이즈 제거) 효과 격리 실험 — 최적 크기 G2 고정

크기 격자(G1~G4)에서 두 언어를 가장 균형 있게 살린 **G2_ko4_en3(한 540/80, 영 620/90)**를 고정하고,
그 위에 **노이즈 제거(머리말·꼬리말·페이지번호)**만 켠 `G5_ko4_en3_clean`을 추가해 비교한다.
크기는 동일하고 **클렌징 여부만 다르므로**, 두 전략의 차이가 곧 클렌징의 순수 효과다.

- 클렌징은 Cell 6의 `clean_documents()` 사용 — **페이지 맨 위/아래 줄에 한정한 위치 기반** 제거라,
  v1에서 본문 섹션 헤더("국민 암예방 수칙 실천지침")까지 지웠던 실수를 방지한다.
- G1~G4는 이미 채점됐으므로 **재실행하지 않는다.** 이 셀들만 실행하면 G5만 새로 인덱싱·채점된다.

In [12]:
# --- 클린 config 추가 (G2 크기 고정 + 노이즈 제거) ---
CLEAN_NAME = "G5_ko4_en3_clean"
STRATEGIES[CLEAN_NAME] = {
    "kind": "cleaned",  # Cell 8 build_chunks가 clean_documents 적용 후 언어별 분할
    "size_by_lang":    {"ko": 540, "en": 620, "unknown": 580},  # G2와 동일
    "overlap_by_lang": {"ko": 80,  "en": 90,  "unknown": 85},   # G2와 동일
}

# 청크 생성(클렌징 적용) — 크기가 G2와 같아도 내용이 달라 chunk 수가 줄어든다
chunk_store[CLEAN_NAME] = build_chunks(CLEAN_NAME)
_avg = sum(len(c.page_content) for c in chunk_store[CLEAN_NAME]) / max(len(chunk_store[CLEAN_NAME]), 1)
print(f"{CLEAN_NAME}  chunk수={len(chunk_store[CLEAN_NAME])}  평균길이={_avg:.0f}")
print(f"(참고) G2_ko4_en3  chunk수={len(chunk_store['G2_ko4_en3'])}")

# 실행 + 채점 (run_strategy가 인덱싱·생성·RAGAS·CSV저장까지 처리)
score_tables[CLEAN_NAME] = run_strategy(CLEAN_NAME)
print(f"{CLEAN_NAME}: 완료")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


제거된 머리말/꼬리말 줄 수(누적): 1560
G5_ko4_en3_clean  chunk수=2704  평균길이=475
(참고) G2_ko4_en3  chunk수=2773
  G5_ko4_en3_clean: 신규 인덱싱 2704개 ... (CPU면 시간 소요)


RAG[G5_ko4_en3_clean]: 100%|██████████| 30/30 [01:27<00:00,  2.91s/it]


Evaluating:   0%|          | 0/90 [00:00<?, ?it/s]

Exception raised in Job[89]: BadRequestError(Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Cd9uSsMrkxQvrsiv7CZVz'})


G5_ko4_en3_clean: 완료


In [13]:
# --- 클렌징 효과 격리: G2(클린 off) vs G5(클린 on), 크기는 동일 ---
import re
def _lg(q): return "KO" if re.search("[가-힣]", str(q)) else "EN"

g2 = score_tables["G2_ko4_en3"].copy(); g2["_lang"] = g2["user_input"].apply(_lg)
g5 = score_tables[CLEAN_NAME].copy();   g5["_lang"] = g5["user_input"].apply(_lg)

print("=== 클렌징 효과 (G2 크기 고정, 클린 off vs on) ===")
print(f"chunk수: G2={len(chunk_store['G2_ko4_en3'])}  ->  G5(clean)={len(chunk_store[CLEAN_NAME])}")
print()
print(f"{'지표':18s} {'G2(off)':>9s} {'G5(on)':>9s} {'델타':>9s}")
for col in ["context_precision", "faithfulness", "answer_relevancy"]:
    a, b = g2[col].mean(), g5[col].mean()
    print(f"{col:18s} {a:9.4f} {b:9.4f} {b-a:+9.4f}")

print("\n--- 언어별 context_precision ---")
for L in ["KO", "EN"]:
    a = g2[g2["_lang"]==L]["context_precision"].mean()
    b = g5[g5["_lang"]==L]["context_precision"].mean()
    print(f"  [{L}] G2={a:.4f}  G5(clean)={b:.4f}  델타={b-a:+.4f}")

# v1 교훈 확인: 클렌징이 특정 한국어 문항을 죽였는지 (특히 식이요법)
print("\n--- 클렌징으로 CP가 가장 떨어진 한국어 문항 (v1 헤더삭제 재발 점검) ---")
m = g2[g2["_lang"]=="KO"][["user_input","context_precision"]].merge(
    g5[g5["_lang"]=="KO"][["user_input","context_precision"]],
    on="user_input", suffixes=("_off","_on"))
m["델타"] = (m["context_precision_on"] - m["context_precision_off"]).round(3)
for _, r in m.sort_values("델타").head(5).iterrows():
    print(f"  {r['델타']:+.2f}  (off {r['context_precision_off']:.2f} -> on {r['context_precision_on']:.2f})  {r['user_input']}")

=== 클렌징 효과 (G2 크기 고정, 클린 off vs on) ===
chunk수: G2=2773  ->  G5(clean)=2704

지표                   G2(off)    G5(on)        델타
context_precision     0.6773    0.6150   -0.0623
faithfulness          0.6561    0.5981   -0.0581
answer_relevancy      0.9039    0.8702   -0.0337

--- 언어별 context_precision ---
  [KO] G2=0.6804  G5(clean)=0.6162  델타=-0.0642
  [EN] G2=0.6711  G5(clean)=0.6123  델타=-0.0588

--- 클렌징으로 CP가 가장 떨어진 한국어 문항 (v1 헤더삭제 재발 점검) ---
  -0.50  (off 1.00 -> on 0.50)  유방절제술 후 재건수술에는 어떤 방법이 있나요?
  -0.50  (off 0.50 -> on 0.00)  유전성 유방암은 전체 유방암의 어느 정도이며 어떤 유전자와 관련되나요?
  -0.50  (off 1.00 -> on 0.50)  폐경 전 환자와 폐경 후 환자의 항호르몬 치료는 어떻게 다른가요?
  -0.42  (off 1.00 -> on 0.58)  유방암 검진은 몇 살부터 받는 것이 권장되나요?
  -0.33  (off 0.32 -> on 0.00)  BRCA 유전자 검사는 누구에게 권장되나요?
